In [1]:
%pip install beautifulsoup4

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv, time, os

headers = {"User-Agent": "final year academic project"}

testing on the one plant

In [ ]:
url = "https://slsdb.manipal.edu/cdmmm/med_plants_res.php?Plant_ID=PL0001"
r = requests.get(url, headers=headers)
r.encoding = r.apparent_encoding   
soup = BeautifulSoup(r.text, "html.parser")

tables = soup.find_all("table")
print("number of tables:", len(tables))
print(soup.find(["h1","h2","h3"]).get_text(strip=True))

In [ ]:
for t in tables[:3]:
    for row in t.find_all("tr"):
        cells = [c.get_text(" ", strip=True) for c in row.find_all(["td","th"])]
        if any(cells):
            print(cells)
    print("----")

In [ ]:
def get_page(url):
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
    except Exception as e:
        print("failed:", url, e)
        return None
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    tables = []
    for t in soup.find_all("table"):
        rows = []
        for tr in t.find_all("tr"):
            cells = [c.get_text(" ", strip=True) for c in tr.find_all(["td","th"])]
            if any(cells):
                rows.append(cells)
        if rows:
            tables.append(rows)

    # links (compounds, targets, diseases)
    links = []
    for a in soup.find_all("a", href=True):
        h = a["href"]
        if "_ID=" in h or "target" in h.lower() or "disease" in h.lower():
            links.append((a.get_text(" ", strip=True), h))

    title = soup.find(["h1","h2","h3"])
    title = title.get_text(" ", strip=True) if title else ""
    return title, tables, links

In [ ]:
os.makedirs("cdmmm_out", exist_ok=True)

def scrape(kind, ids, make_url, delay=0.5):
    idx_file  = f"cdmmm_out/{kind}_index.csv"
    tab_file  = f"cdmmm_out/{kind}_tables.csv"
    link_file = f"cdmmm_out/{kind}_links.csv"

    done = set()
    if os.path.exists(idx_file):
        with open(idx_file, encoding="utf-8") as f:
            next(f, None)
            for line in f:
                done.add(line.split(",")[0])

    idx  = open(idx_file,  "a", newline="", encoding="utf-8")
    tab  = open(tab_file,  "a", newline="", encoding="utf-8")
    lnk  = open(link_file, "a", newline="", encoding="utf-8")
    idx_w, tab_w, lnk_w = csv.writer(idx), csv.writer(tab), csv.writer(lnk)
    if idx.tell() == 0:
        idx_w.writerow(["id", "title", "url"])
        tab_w.writerow(["id", "table_no", "row_no", "cells"])
        lnk_w.writerow(["id", "text", "href"])

    for i, rid in enumerate(ids, 1):
        if rid in done:
            continue
        url = make_url(rid)
        result = get_page(url)
        if result is None:
            continue
        title, tables, links = result
        idx_w.writerow([rid, title, url])
        for tn, table in enumerate(tables):
            for rn, row in enumerate(table):
                tab_w.writerow([rid, tn, rn, " | ".join(row)])
        for text, href in links:
            lnk_w.writerow([rid, text, href])

        if i % 25 == 0:
            print(kind, i, "/", len(ids))
            idx.flush(); tab.flush(); lnk.flush()
        time.sleep(delay)  

    idx.close(); tab.close(); lnk.close()
    print("done:", kind)

plants 

In [ ]:
plant_ids = ["PL%04d" % n for n in range(1, 68)]
scrape("plants", plant_ids,
       lambda x: "https://slsdb.manipal.edu/cdmmm/med_plants_res.php?Plant_ID=" + x)

compounds

In [ ]:
compound_ids = ["CDAMM%05d" % n for n in range(1, 3034)]
scrape("compounds", compound_ids,
       lambda x: "https://slsdb.manipal.edu/cdmmm/compound_res.php?Compound_ID=" + x)

In [ ]:
import pandas as pd
df = pd.read_csv("cdmmm_out/compounds_tables.csv")
print(df.shape)
df.head(15)